# 07 - Model Training for Rainfall Forecasting

This notebook trains the three rainfall models used by the web app. It uses only the model-ready files exported by notebook 05:

- `data/processed/model_ready/sea_rainfall_daily_2020_2025_model_ready_strict_forecast_X.csv`
- `data/processed/model_ready/sea_rainfall_daily_2020_2025_model_ready_targets_y.csv`

The previous strict autoregressive models were too smooth for multi-day future forecasts. This rebuilt version adds weather-assisted features derived from the model-ready target table:

- Open-Meteo rainfall is used as a historical proxy for an external web forecast feature.
- NASA train-only day-of-year climatology is used as a seasonal baseline feature.
- The deployment web app fills these same feature names with future web forecast and NASA seasonal baseline values.

The selected target remains `target_nasa_power_precipitation_mm`.

In [ ]:
from __future__ import annotations

import json
import math
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

X_PATH = PROJECT_ROOT / "data" / "processed" / "model_ready" / "sea_rainfall_daily_2020_2025_model_ready_strict_forecast_X.csv"
Y_PATH = PROJECT_ROOT / "data" / "processed" / "model_ready" / "sea_rainfall_daily_2020_2025_model_ready_targets_y.csv"
MODEL_DIR = PROJECT_ROOT / "models" / "07_model_training"
REPORT_DIR = PROJECT_ROOT / "reports" / "07_model_training"
TABLE_DIR = REPORT_DIR / "tables"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "target_nasa_power_precipitation_mm"
PROVIDER_PROXY_COLUMN = "target_open_meteo_precipitation_mm"
PREDICTION_CAP_MM = 500.0
WET_DAY_THRESHOLD_MM = 1.0
RANDOM_STATE = 42
INTENSITY_WEIGHT_SCALE = 0.25

print("Project root:", PROJECT_ROOT)
print("X:", X_PATH)
print("Y:", Y_PATH)

## Load Model-Ready Data

The training table is assembled from model-ready `X` and model-ready `y`. No raw collection files are used here.

In [ ]:
X = pd.read_csv(X_PATH, parse_dates=["date"])
y = pd.read_csv(Y_PATH, parse_dates=["date"])

required_y = ["sample_id", "entity_id", "date", "split", TARGET_COLUMN, PROVIDER_PROXY_COLUMN]
missing_y = [column for column in required_y if column not in y.columns]
if missing_y:
    raise KeyError(f"Missing model-ready target columns: {missing_y}")

data = X.merge(
    y[["sample_id", TARGET_COLUMN, PROVIDER_PROXY_COLUMN]],
    on="sample_id",
    how="inner",
    validate="one_to_one",
)
data["day_of_year"] = data["date"].dt.dayofyear

base_feature_columns = ["entity_id"] + [column for column in X.columns if column.startswith("feature_")]

summary = data.groupby("split")[TARGET_COLUMN].agg(rows="count", mean_mm_day="mean", std_mm_day="std", max_mm_day="max").reset_index()
summary["wet_day_ratio"] = data.groupby("split")[TARGET_COLUMN].apply(lambda values: float((values >= WET_DAY_THRESHOLD_MM).mean())).values
display(summary)

## Weather-Assisted Feature Engineering

These features are still built from model-ready files:

- `target_open_meteo_precipitation_mm` becomes a historical proxy for future web forecast rainfall.
- NASA seasonal baseline is computed from train split only, then applied to validation/test/future dates.

This makes the model less flat because it can react to an external forecast signal at inference time.

In [ ]:
def provider_probability_proxy(provider_mm: pd.Series) -> np.ndarray:
    values = provider_mm.fillna(0.0).to_numpy(dtype=float)
    return np.select(
        [values >= 25.0, values >= 10.0, values >= 1.0],
        [95.0, 80.0, 60.0],
        default=10.0,
    )

train_rows = data["split"].eq("train")
train_climatology = (
    data.loc[train_rows]
    .groupby(["entity_id", "day_of_year"], as_index=False)[TARGET_COLUMN]
    .mean()
    .rename(columns={TARGET_COLUMN: "feature_nasa_seasonal_baseline_mm"})
)
city_climatology = (
    data.loc[train_rows]
    .groupby("entity_id", as_index=False)[TARGET_COLUMN]
    .mean()
    .rename(columns={TARGET_COLUMN: "city_train_target_mean_mm"})
)

data = data.merge(train_climatology, on=["entity_id", "day_of_year"], how="left")
data = data.merge(city_climatology, on="entity_id", how="left")
data["feature_nasa_seasonal_baseline_mm"] = data["feature_nasa_seasonal_baseline_mm"].fillna(data["city_train_target_mean_mm"])

data["feature_provider_forecast_precipitation_mm"] = data[PROVIDER_PROXY_COLUMN].fillna(0.0).clip(lower=0.0)
data["feature_provider_forecast_log1p_precipitation"] = np.log1p(data["feature_provider_forecast_precipitation_mm"])
data["feature_provider_forecast_probability_pct"] = provider_probability_proxy(data["feature_provider_forecast_precipitation_mm"])
data["feature_provider_forecast_is_wet"] = (data["feature_provider_forecast_precipitation_mm"] >= WET_DAY_THRESHOLD_MM).astype(float)
data["feature_provider_minus_nasa_baseline_mm"] = data["feature_provider_forecast_precipitation_mm"] - data["feature_nasa_seasonal_baseline_mm"]

weather_assisted_features = [
    "feature_provider_forecast_precipitation_mm",
    "feature_provider_forecast_log1p_precipitation",
    "feature_provider_forecast_probability_pct",
    "feature_provider_forecast_is_wet",
    "feature_nasa_seasonal_baseline_mm",
    "feature_provider_minus_nasa_baseline_mm",
]
feature_columns = base_feature_columns + weather_assisted_features

feature_preview = data[["sample_id", "split", TARGET_COLUMN, PROVIDER_PROXY_COLUMN, *weather_assisted_features]].head(10)
display(feature_preview)

## Leakage Guard

The target being predicted is NASA rainfall. No same-day NASA target is used as a direct feature.

The Open-Meteo column is used only as a provider-forecast proxy. At deployment, the backend fills the same feature names using a future web forecast API response, not the observed target table.

In [ ]:
leakage_audit = pd.DataFrame([
    {
        "column": column,
        "role": (
            "external_forecast_proxy" if column in weather_assisted_features and "provider" in column
            else "train_only_nasa_climatology" if column == "feature_nasa_seasonal_baseline_mm"
            else "strict_forecast_feature"
        ),
        "uses_selected_target_directly": False,
    }
    for column in feature_columns
])

if TARGET_COLUMN in feature_columns:
    raise RuntimeError("Selected target leaked into feature columns.")

leakage_audit.to_csv(TABLE_DIR / "02_leakage_audit.csv", index=False)
display(leakage_audit.tail(10))

## Train Three Models

All models predict `log1p(NASA rainfall)`. The web app converts model output back with `expm1`.

In [ ]:
def make_preprocessor(feature_columns: list[str]) -> ColumnTransformer:
    categorical_columns = ["entity_id"]
    numeric_columns = [column for column in feature_columns if column != "entity_id"]
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    return ColumnTransformer([
        ("city", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
        ("numeric", numeric_pipeline, numeric_columns),
    ], sparse_threshold=0.0, remainder="drop")


def sample_weights(y_mm: pd.Series) -> np.ndarray:
    values = y_mm.to_numpy(dtype=float)
    intensity_boost = (
        0.45 * (values >= 1.0)
        + 0.90 * (values >= 10.0)
        + 1.75 * (values >= 25.0)
        + np.clip(values / 35.0, 0.0, 1.5)
    )
    return 1.0 + INTENSITY_WEIGHT_SCALE * intensity_boost


def model_specs(feature_columns: list[str]) -> dict[str, Pipeline]:
    return {
        "hist_gradient_boosting_log": Pipeline([
            ("preprocess", make_preprocessor(feature_columns)),
            ("model", HistGradientBoostingRegressor(
                loss="squared_error",
                max_iter=850,
                learning_rate=0.035,
                max_leaf_nodes=63,
                min_samples_leaf=8,
                l2_regularization=0.0,
                early_stopping=True,
                validation_fraction=0.16,
                n_iter_no_change=35,
                random_state=RANDOM_STATE,
            )),
        ]),
        "gradient_boosting_log": Pipeline([
            ("preprocess", make_preprocessor(feature_columns)),
            ("model", GradientBoostingRegressor(
                loss="huber",
                n_estimators=850,
                learning_rate=0.03,
                max_depth=4,
                min_samples_leaf=8,
                subsample=0.85,
                validation_fraction=0.15,
                n_iter_no_change=35,
                random_state=RANDOM_STATE,
            )),
        ]),
        "mlp_log": Pipeline([
            ("preprocess", make_preprocessor(feature_columns)),
            ("model", MLPRegressor(
                hidden_layer_sizes=(192, 96),
                activation="relu",
                solver="adam",
                alpha=1e-5,
                learning_rate_init=0.001,
                batch_size=512,
                max_iter=650,
                early_stopping=True,
                validation_fraction=0.16,
                n_iter_no_change=35,
                random_state=RANDOM_STATE,
            )),
        ]),
    }


def predict_mm(model: Pipeline, X_split: pd.DataFrame) -> np.ndarray:
    max_log = math.log1p(PREDICTION_CAP_MM)
    log_prediction = np.asarray(model.predict(X_split), dtype=float)
    log_prediction = np.nan_to_num(log_prediction, nan=0.0, posinf=max_log, neginf=0.0)
    return np.expm1(np.clip(log_prediction, 0.0, max_log)).clip(0.0, PREDICTION_CAP_MM)


def split_metrics(model: Pipeline, X_split: pd.DataFrame, y_split_mm: pd.Series, split_name: str, model_name: str) -> dict:
    prediction = predict_mm(model, X_split)
    actual = y_split_mm.to_numpy(dtype=float)
    wet_true = actual >= WET_DAY_THRESHOLD_MM
    wet_pred = prediction >= WET_DAY_THRESHOLD_MM
    target_std = float(np.std(actual))
    pred_std = float(np.std(prediction))
    return {
        "model_name": model_name,
        "split": split_name,
        "rows": int(len(actual)),
        "mae_mm_day": float(mean_absolute_error(actual, prediction)),
        "rmse_mm_day": float(mean_squared_error(actual, prediction) ** 0.5),
        "r2": float(r2_score(actual, prediction)),
        "wet_day_accuracy": float(np.mean(wet_true == wet_pred)),
        "target_mean_mm_day": float(np.mean(actual)),
        "prediction_mean_mm_day": float(np.mean(prediction)),
        "target_std_mm_day": target_std,
        "prediction_std_mm_day": pred_std,
        "std_capture_ratio": pred_std / target_std if target_std else None,
        "max_prediction_mm_day": float(np.max(prediction)),
    }

In [ ]:
split_frames = {split: frame.copy() for split, frame in data.groupby("split", sort=False)}
X_train = split_frames["train"][feature_columns]
y_train_mm = split_frames["train"][TARGET_COLUMN]
y_train_log = np.log1p(y_train_mm.clip(lower=0.0))
weights = sample_weights(y_train_mm)

all_metrics = []
artifacts = []
best_model = None
best_model_name = None
best_selection_score = float("inf")
models = model_specs(feature_columns)

for model_name, model in models.items():
    print(f"Training {model_name} ...")
    model.fit(X_train, y_train_log, model__sample_weight=weights)
    model_metrics = []
    for split_name, frame in split_frames.items():
        model_metrics.append(split_metrics(model, frame[feature_columns], frame[TARGET_COLUMN], split_name, model_name))
    metrics_frame = pd.DataFrame(model_metrics)
    all_metrics.append(metrics_frame)

    model_path = MODEL_DIR / f"{model_name}_log_target_pipeline.joblib"
    joblib.dump(model, model_path)

    validation = metrics_frame[metrics_frame["split"].eq("validation")].iloc[0]
    selection_score = validation["mae_mm_day"] - 0.20 * validation["std_capture_ratio"]
    if selection_score < best_selection_score:
        best_selection_score = float(selection_score)
        best_model = model
        best_model_name = model_name

    metadata = {
        "model_name": model_name,
        "selected_target": TARGET_COLUMN,
        "target_transform": "log1p during training; expm1 during prediction",
        "prediction_clip_mm_day": [0.0, PREDICTION_CAP_MM],
        "training_strategy": "weather-assisted model-ready training using Open-Meteo target as future web-forecast proxy",
        "feature_columns": feature_columns,
        "weather_assisted_features": weather_assisted_features,
        "validation_mae_mm_day": float(validation["mae_mm_day"]),
        "validation_std_capture_ratio": float(validation["std_capture_ratio"]),
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (MODEL_DIR / f"{model_name}_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    artifacts.append({"model_name": model_name, "model_path": str(model_path.relative_to(PROJECT_ROOT))})
    print(f"  validation MAE={validation['mae_mm_day']:.4f}; std capture={validation['std_capture_ratio']:.3f}")

metrics = pd.concat(all_metrics, ignore_index=True)
best_metrics = metrics[metrics["model_name"].eq(best_model_name)].copy()
joblib.dump(best_model, MODEL_DIR / "best_model_log_target_pipeline.joblib")

best_metadata = json.loads((MODEL_DIR / f"{best_model_name}_metadata.json").read_text(encoding="utf-8"))
best_metadata.update({
    "best_model_name": best_model_name,
    "selection_metric": "validation MAE with small std-capture tie adjustment",
    "selected_as_best": True,
})
(MODEL_DIR / "best_model_metadata.json").write_text(json.dumps(best_metadata, indent=2), encoding="utf-8")

display(metrics.sort_values(["model_name", "split"]))

## Save Reports And Predictions

In [ ]:
prediction_rows = []
for split_name, frame in split_frames.items():
    prediction = predict_mm(best_model, frame[feature_columns])
    prediction_rows.append(pd.DataFrame({
        "sample_id": frame["sample_id"].to_numpy(),
        "entity_id": frame["entity_id"].to_numpy(),
        "date": frame["date"].dt.strftime("%Y-%m-%d").to_numpy(),
        "split": split_name,
        "actual_mm": frame[TARGET_COLUMN].to_numpy(dtype=float),
        "prediction_mm": prediction,
        "absolute_error_mm": np.abs(frame[TARGET_COLUMN].to_numpy(dtype=float) - prediction),
    }))
predictions_for_best = pd.concat(prediction_rows, ignore_index=True)

summary.to_csv(TABLE_DIR / "01_training_data_summary.csv", index=False)
leakage_audit.to_csv(TABLE_DIR / "02_leakage_audit.csv", index=False)
metrics[metrics["split"].eq("validation")].assign(trial_id=lambda df: df["model_name"] + "_weather_assisted").to_csv(TABLE_DIR / "03_hyperparameter_trials.csv", index=False)
metrics.to_csv(TABLE_DIR / "04_all_trial_metrics_by_split.csv", index=False)
best_metrics.to_csv(TABLE_DIR / "05_best_model_metrics_by_split.csv", index=False)
predictions_for_best.to_csv(TABLE_DIR / "06_best_model_predictions_by_split.csv", index=False)
pd.DataFrame(artifacts).to_csv(TABLE_DIR / "07_model_artifacts.csv", index=False)

test_city_errors = (
    predictions_for_best[predictions_for_best["split"].eq("test")]
    .groupby("entity_id")["absolute_error_mm"]
    .agg(rows="count", mae_mm_day="mean", max_error_mm_day="max")
    .reset_index()
)
test_city_errors.to_csv(TABLE_DIR / "08_best_model_test_error_by_city.csv", index=False)

best_val = best_metrics[best_metrics["split"].eq("validation")].iloc[0]
best_test = best_metrics[best_metrics["split"].eq("test")].iloc[0]
summary_text = f"""# Model Training Summary

## Selected Target
- `{TARGET_COLUMN}`.
- The target choice comes from notebook 06.

## Data Inputs
- `sea_rainfall_daily_2020_2025_model_ready_strict_forecast_X.csv`.
- `sea_rainfall_daily_2020_2025_model_ready_targets_y.csv`.
- No raw collection files are used in this notebook.

## Retraining Change
- Retrained on {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}.
- Added weather-assisted model-ready features so the web forecast series does not collapse into an overly flat recursive line.
- Open-Meteo target is used during training as a proxy for the future web forecast feature used by the frontend/backend.
- NASA seasonal baseline is computed from the train split only.

## Leakage Control
- The selected NASA target column is not used as a direct feature.
- Station ground truth is not used as a model feature.
- Future deployment fills provider features from web forecast, not from the observed target table.

## Models Trained
- `hist_gradient_boosting_log`.
- `gradient_boosting_log`.
- `mlp_log`.

## Best Model
- Best model: `{best_model_name}`.
- Validation MAE: {best_val['mae_mm_day']:.4f} mm/day.
- Validation prediction std capture: {best_val['std_capture_ratio']:.3f}.
- Test MAE: {best_test['mae_mm_day']:.4f} mm/day.
- Test RMSE: {best_test['rmse_mm_day']:.4f} mm/day.
- Test R2: {best_test['r2']:.4f}.
- Test wet-day accuracy: {best_test['wet_day_accuracy']:.4f}.
- Test prediction std capture: {best_test['std_capture_ratio']:.3f}.

## Saved Outputs
- Models: `models/07_model_training/`.
- Tables: `reports/07_model_training/tables/`.

## Important Prediction Note
The saved pipelines predict `log1p(rainfall_mm)`. For inference, convert prediction back with `expm1`, then clip to `[0, 500]` mm/day.
"""
(REPORT_DIR / "MODEL_TRAINING_SUMMARY.md").write_text(summary_text, encoding="utf-8")

print("Best model:", best_model_name)
display(best_metrics)
print("Saved models to", MODEL_DIR)
print("Saved reports to", TABLE_DIR)